## РЕГРЕССИЯ — ОПТИМИЗАЦИЯ ГИПЕРПАРАМЕТРОВ

### Цель
Для каждого таргета (IC50, CC50, SI) взять топ-2 модели из baseline и оптимизировать их гиперпараметры с помощью RandomizedSearchCV.


In [1]:
import sys
import os
import pandas as pd
import numpy as np
import joblib

# Добавляем корень проекта в путь
sys.path.append(os.path.dirname(os.getcwd()))

from src import get_param_grids

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score
)

## 1. Загрузка данных и моделей

**Загружаемые данные:**
- `X_train_scaled.pkl` / `X_test_scaled.pkl` - признаки (масштабированные)
- `y_train_reg.pkl` / `y_test_reg.pkl` - регрессионные таргеты

**Загружаемые артефакты:**
- `top_models_regression.pkl` - словарь с топ-2 моделями для каждого таргета (из ноутбука 03)
- `param_grids` - сетки гиперпараметров для каждой модели (из src.models)

In [2]:
SAVE_PATH = '../data/processed/'

X_train = joblib.load(f'{SAVE_PATH}X_train_scaled.pkl')
X_test = joblib.load(f'{SAVE_PATH}X_test_scaled.pkl')
y_train_reg = joblib.load(f'{SAVE_PATH}y_train_reg.pkl')
y_test_reg = joblib.load(f'{SAVE_PATH}y_test_reg.pkl')

# Загружаем топ-модели
top_models_reg = joblib.load('../artifacts/top_models_regression.pkl')
param_grids = get_param_grids()

## 2. Настройка параметров оптимизации

**Целевые переменные:**
- `pIC50` → IC50 (ингибирующая активность)
- `pCC50` → CC50 (цитотоксичность)
- `log_SI` → SI (селективность)

**Параметры оптимизации:**
- Метод: RandomizedSearchCV
- Количество итераций: 20
- Кросс-валидация: 5-fold
- Метрика: R²
- Random state: 42 (для воспроизводимости)

In [3]:
REGRESSION_TARGETS = ['pIC50', 'pCC50', 'log_SI']
TARGET_NAMES = ['IC50', 'CC50', 'SI']

# Структура для результатов
best_models = {}
tuned_results = {target: {} for target in TARGET_NAMES}

## 3. Оптимизация гиперпараметров

Для каждого таргета:
1. Берём топ-2 модели из baseline
2. Для каждой модели запускаем RandomizedSearchCV
3. Выбираем модель с лучшим R² на тестовой выборке
4. Сохраняем лучшую модель

In [4]:
for i, target_col in enumerate(REGRESSION_TARGETS):
    target_name = TARGET_NAMES[i]
    
    print(f"\nТАРГЕТ: {target_name} ({target_col})")
    print('─'*60)
    
    y_train = y_train_reg[target_col]
    y_test = y_test_reg[target_col]
    
    # Топ-2 модели для этого таргета
    top_2 = top_models_reg[target_name]
    
    baseline_df = pd.read_csv(f'../artifacts/results/regression_baseline_{target_name}.csv', index_col=0)
    
    best_score = -np.inf
    best_model = None
    
    for model_name in top_2:
        print(f"Оптимизация: {model_name}")
        
        # Получаем модель из словаря (нужно создать)
        # Временно создаём модели здесь
        if 'RandomForest' in model_name:
            from sklearn.ensemble import RandomForestRegressor
            base_model = RandomForestRegressor(random_state=42)
        elif 'XGBoost' in model_name:
            from xgboost import XGBRegressor
            base_model = XGBRegressor(random_state=42, verbosity=0)
        elif 'GradientBoosting' in model_name:
            from sklearn.ensemble import GradientBoostingRegressor
            base_model = GradientBoostingRegressor(random_state=42)
        else:
            continue
        
        
        # Подбираем параметры
        rs = RandomizedSearchCV(
            base_model,
            param_grids.get(model_name, {}),
            n_iter=20,
            cv=5,
            scoring='r2',
            random_state=42,
            n_jobs=-1,
            verbose=0
        )
        rs.fit(X_train, y_train)
        
        print(f"    Лучшие параметры: {rs.best_params_}")
        y_pred = rs.predict(X_test)
        test_r2 = r2_score(y_test, y_pred)
        print(f"    CV R²: {rs.best_score_:.4f}    Test R²: {test_r2:.4f}")
        baseline_r2 = baseline_df.loc[model_name, 'R2']
        
        
        
        # Сохраняем результаты
        tuned_results[target_name][model_name] = {
            'R2': test_r2,
            'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
            'MAE': mean_absolute_error(y_test, y_pred),
            'best_params': rs.best_params_
        }
        
        if baseline_r2 > test_r2:
            # Используем baseline модель (нужно создать заново)
            if 'RandomForest' in model_name:
                best_for_model = RandomForestRegressor(random_state=42)
            elif 'XGBoost' in model_name:
                best_for_model = XGBRegressor(random_state=42, verbosity=0)
            elif 'GradientBoosting' in model_name:
                best_for_model = GradientBoostingRegressor(random_state=42)
            best_for_model.fit(X_train, y_train)
            
        if baseline_r2 > best_score:
                best_score = baseline_r2
                best_model = best_for_model

        else:
            if test_r2 > best_score:
                best_score = test_r2
                best_model = rs.best_estimator_
  

    
    # Сохраняем лучшую модель
    best_models[target_name] = best_model
    
    # Сохраняем модель на диск
    joblib.dump(best_model, f'../artifacts/models/best_{target_name}_regressor.pkl')
    
    print(f"\nЛучшая модель для {target_name}: {best_model.__class__.__name__}   Test R²: {best_score:.4f}")


ТАРГЕТ: IC50 (pIC50)
────────────────────────────────────────────────────────────
Оптимизация: GradientBoosting
    Лучшие параметры: {'n_estimators': 100, 'min_samples_split': 5, 'max_depth': 3, 'learning_rate': 0.05}
    CV R²: 0.4532    Test R²: 0.4826
Оптимизация: RandomForest
    Лучшие параметры: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 20}
    CV R²: 0.4323    Test R²: 0.4934

Лучшая модель для IC50: GradientBoostingRegressor   Test R²: 0.5211

ТАРГЕТ: CC50 (pCC50)
────────────────────────────────────────────────────────────
Оптимизация: RandomForest
    Лучшие параметры: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 20}
    CV R²: 0.4463    Test R²: 0.3777
Оптимизация: GradientBoosting
    Лучшие параметры: {'n_estimators': 100, 'min_samples_split': 10, 'max_depth': 3, 'learning_rate': 0.1}
    CV R²: 0.4398    Test R²: 0.3612

Лучшая модель для CC50: RandomForestRegressor   Test R²: 0.3816

ТАРГЕТ: 

## 4. Результаты оптимизации

В таблицах ниже представлены метрики качества для оптимизированных моделей.
- **R²** - основной критерий качества (чем выше, тем лучше)
- **RMSE** - среднеквадратичная ошибка (чем ниже, тем лучше)
- **MAE** - средняя абсолютная ошибка (чем ниже, тем лучше)
- **best_params** - оптимальные гиперпараметры

In [7]:
for target_name in TARGET_NAMES:
    print(f"{target_name} — Результаты оптимизации")
    print('─'*60)
    
    df = pd.DataFrame(tuned_results[target_name]).T
    display(df.round(4))
    df.to_csv(f'../artifacts/results/regression_tuned_{target_name}.csv')

IC50 — Результаты оптимизации
────────────────────────────────────────────────────────────


,R2,RMSE,MAE,best_params
GradientBoosting,0.482576,0.729579,0.584159,"{'n_estimators': 100, 'min_samples_split': 5, ..."
RandomForest,0.493427,0.721889,0.553058,"{'n_estimators': 200, 'min_samples_split': 10,..."


CC50 — Результаты оптимизации
────────────────────────────────────────────────────────────


,R2,RMSE,MAE,best_params
RandomForest,0.377708,0.554608,0.378899,"{'n_estimators': 100, 'min_samples_split': 5, ..."
GradientBoosting,0.361229,0.561903,0.396178,"{'n_estimators': 100, 'min_samples_split': 10,..."


SI — Результаты оптимизации
────────────────────────────────────────────────────────────


,R2,RMSE,MAE,best_params
RandomForest,0.29066,0.699753,0.518308,"{'n_estimators': 50, 'min_samples_split': 2, '..."
XGBoost,0.300035,0.695113,0.516182,"{'subsample': 0.8, 'n_estimators': 200, 'max_d..."


## 5. Сравнение с baseline

Анализируем, удалось ли улучшить качество моделей.
- **Baseline R²** - качество модели до оптимизации
- **Tuned R²** - качество модели после оптимизации
- **Улучшение** = Tuned R² - Baseline R²

Положительное значение означает улучшение.

In [8]:
comparison = {}

for target_name in TARGET_NAMES:
    # Загружаем baseline результаты
    baseline_df = pd.read_csv(f'../artifacts/results/regression_baseline_{target_name}.csv', index_col=0)
    
    # Топ-2 модели
    top_2 = top_models_reg[target_name]
    
    comparison[target_name] = {}
    for model_name in top_2:
        baseline_r2 = baseline_df.loc[model_name, 'R2']
        tuned_r2 = tuned_results[target_name][model_name]['R2']
        improvement = tuned_r2 - baseline_r2
        
        comparison[target_name][model_name] = {
            'Baseline R²': baseline_r2,
            'Tuned R²': tuned_r2,
            'Improvement': improvement
        }
        
        print(f"\n{target_name} — {model_name}:")
        print(f"  Baseline R²: {baseline_r2:.4f}    Tuned R²:{tuned_r2:.4f}     Улучшение:{improvement:.4f}")

# Сохраняем сравнение
joblib.dump(comparison, '../artifacts/results/regression_comparison.pkl')


IC50 — GradientBoosting:
  Baseline R²: 0.5211    Tuned R²:0.4826     Улучшение:-0.0385

IC50 — RandomForest:
  Baseline R²: 0.5106    Tuned R²:0.4934     Улучшение:-0.0171

CC50 — RandomForest:
  Baseline R²: 0.3816    Tuned R²:0.3777     Улучшение:-0.0039

CC50 — GradientBoosting:
  Baseline R²: 0.3549    Tuned R²:0.3612     Улучшение:0.0064

SI — RandomForest:
  Baseline R²: 0.3057    Tuned R²:0.2907     Улучшение:-0.0150

SI — XGBoost:
  Baseline R²: 0.2867    Tuned R²:0.3000     Улучшение:0.0133


['../artifacts/results/regression_comparison.pkl']

## Заключение

Оптимизация гиперпараметров не дала значимого улучшения — изменения в пределах статистического шума (±0.01-0.02 R²). Это указывает, что потенциал роста лежит не в настройке параметров, а в других направлениях.

**Рекомендации:**
1. Отбор признаков и создание новых дескрипторов
2. Использование LightGBM/CatBoost и ансамблей (Stacking/Voting)
3. Увеличение итераций RandomizedSearchCV до 100+

**Итоговые модели (готовы к использованию):**
- `best_IC50_regressor.pkl` — RandomForest (R² = 0.493)
- `best_CC50_regressor.pkl` — RandomForest (R² = 0.378)
- `best_SI_regressor.pkl` — XGBoost (R² = 0.300)

**Вывод:** Модели достигли предела для текущих данных и признаков. Дальнейший прогресс требует работы с данными и архитектурой, а не перебора гиперпараметров.